# Image Classification Using Convolutional Neural Networks (CNNs)
## Mini Project - Computer Vision Fundamentals

**Objective:** Build a CNN-based image classifier to classify images into multiple categories

**Dataset:** Cats vs Dogs (or any other dataset of your choice)

## 1. Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import os

print(f"TensorFlow version: {tf.__version__}")

## 2. Data Preparation
### Download and Setup Dataset

In [ ]:
# Download Cats vs Dogs dataset
dataset_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
path_to_zip = keras.utils.get_file('cats_and_dogs.zip', origin=dataset_url, extract=True)
base_dir = os.path.join(os.path.dirname(path_to_zip), 'PetImages')

print(f"Dataset downloaded to: {base_dir}")

In [ ]:
# Remove corrupted images
import os
num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join(base_dir, folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = tf.compat.as_bytes("JFIF") in fobj.peek(10)
        finally:
            fobj.close()
        if not is_jfif:
            num_skipped += 1
            os.remove(fpath)

print(f"Deleted {num_skipped} corrupted images")

### Configure Data Parameters

In [ ]:
# Hyperparameters
IMG_SIZE = 150
BATCH_SIZE = 32
EPOCHS = 20
VALIDATION_SPLIT = 0.2

### Data Augmentation and Preprocessing

In [ ]:
# Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT
)

# Validation data (only rescaling)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

validation_generator = val_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")

## 3. Model Building
### Create CNN Architecture

In [ ]:
model = models.Sequential([
    # First Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Third Convolutional Block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Fourth Convolutional Block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Flatten and Dense Layers
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

### Compile Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 4. Model Training

In [ ]:
# Early stopping callback
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=[early_stop]
)

## 5. Model Evaluation
### Plot Training History

In [ ]:
# Plot accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

### Evaluate on Validation Set

In [ ]:
val_loss, val_accuracy = model.evaluate(validation_generator)
print(f"\nValidation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

### Make Predictions on Sample Images

In [ ]:
# Get a batch of validation images
sample_images, sample_labels = next(validation_generator)

# Make predictions
predictions = model.predict(sample_images)

# Display results
plt.figure(figsize=(15, 10))
for i in range(min(9, len(sample_images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i])
    predicted_class = "Dog" if predictions[i] > 0.5 else "Cat"
    actual_class = "Dog" if sample_labels[i] > 0.5 else "Cat"
    color = "green" if predicted_class == actual_class else "red"
    plt.title(f"Pred: {predicted_class} ({predictions[i][0]:.2f})\nActual: {actual_class}", color=color)
    plt.axis('off')

plt.tight_layout()
plt.savefig('predictions_sample.png')
plt.show()

### Test on Custom Image

In [ ]:
def predict_image(image_path):
    img = keras.preprocessing.image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    prediction = model.predict(img_array)[0][0]
    predicted_class = "Dog" if prediction > 0.5 else "Cat"
    confidence = prediction if prediction > 0.5 else 1 - prediction
    
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class} (Confidence: {confidence:.2%})")
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence

# Example usage (uncomment and provide image path):
# predict_image('path/to/your/image.jpg')

## 6. Save Model

In [ ]:
# Save model in .h5 format
model.save('cnn_image_classifier.h5')
print("Model saved as 'cnn_image_classifier.h5'")

# Save model in SavedModel format (recommended)
model.save('cnn_image_classifier_savedmodel')
print("Model saved as 'cnn_image_classifier_savedmodel'")

## 7. Load and Use Saved Model

In [ ]:
# Load the saved model
loaded_model = keras.models.load_model('cnn_image_classifier.h5')
print("Model loaded successfully!")

# Verify it works
test_loss, test_accuracy = loaded_model.evaluate(validation_generator)
print(f"Loaded Model Accuracy: {test_accuracy:.4f}")

## Summary Report

### Learning Outcomes:
1. **Data Preprocessing**: Learned to load, resize, normalize images and apply data augmentation techniques to improve model generalization.
2. **CNN Architecture**: Built a multi-layer CNN with convolution, pooling, and dense layers, understanding how each component extracts and processes visual features.
3. **Training Process**: Successfully trained a deep learning model with proper validation, monitoring accuracy/loss curves, and implementing early stopping.
4. **Model Evaluation**: Evaluated model performance using validation metrics and visualized predictions to interpret results effectively.
5. **Deployment Ready**: Saved the trained model in multiple formats for future use and deployment in real-world applications.

### Key Concepts Mastered:
- Convolutional layers for feature extraction
- Pooling layers for dimensionality reduction
- Dropout for preventing overfitting
- Data augmentation for better generalization
- Binary classification with sigmoid activation
- Model persistence and loading